# 🎙️ NEXUS Voice Clone — your own voice for NEXUS

This notebook makes NEXUS speak in **your voice**, using **Chatterbox** (Resemble AI, open source, MIT licence, English) on Google Colab's **free GPU**. No credit card, no paid API.

**Why not IndicF5?** IndicF5 is made for 11 Indian languages (Assamese, Bengali, Gujarati, Hindi, Kannada, Malayalam, Marathi, Odia, Punjabi, Tamil, Telugu) — English is not one of them. Chatterbox is built for English voice cloning from a short clip, so it fits better.

## First time only (5 minutes)
1. **Turn on the free GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. **Give the notebook your NEXUS voice key** (the same long password you put in Render as `VOICE_CLONE_KEY`): click the 🔑 **key icon** in the left sidebar → **Add new secret** → Name: `NEXUS_VOICE_KEY`, Value: your key → switch **Notebook access** on. Colab keeps it private and remembers it.
3. Menu **Runtime → Run all**. When Cell 3 asks, allow Google Drive and choose your recording — it is saved in your Drive, so you never upload it again.

## Every time after that
Open this notebook → **Runtime → Run all** → wait about 3 minutes. NEXUS finds your voice by itself (no links to copy). Keep the tab open.

If Colab says the session crashed right after Cell 1, that is its planned restart — just **Run all** once more.

Use only **your own** voice (or a voice you have permission to use). Every clip Chatterbox makes carries an inaudible watermark (Resemble AI's "Perth") that marks it as AI-generated.

In [ ]:
# ===== Cell 1: check the GPU and install Chatterbox (first run ≈ 3 minutes; later sessions are faster) =====
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print("✅ GPU found:", gpu.stdout.strip())
else:
    print("⚠️  No GPU. It still works on the CPU, but slowly (a minute or more per sentence).")
    print("   Fix: Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all.")
    print("   (If Colab says no GPU is available, today's free GPU time is used up — try later.)")
MARK = "/content/.nexus_voice_installed_v2"
if os.path.exists(MARK):
    print("✅ Chatterbox is installed.")
else:
    print("Installing chatterbox-tts 0.1.7 and its pinned libraries (PyTorch 2.6 + matching torchvision 0.21) — please wait…")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chatterbox-tts==0.1.7", "torchvision==0.21.0"], capture_output=True, text=True)
    if r.returncode != 0:
        print("❌ The installation failed. Last lines of the error:\\n")
        print("\\n".join((r.stderr or r.stdout).strip().splitlines()[-25:]))
        raise SystemExit("Try: Runtime → Disconnect and delete runtime, then Runtime → Run all.")
    open(MARK, "w").write("ok")
    if "numpy" in sys.modules or "torch" in sys.modules:
        print("✅ Installed. Colab must restart once to use the new libraries — it will say 'session crashed'.")
        print("   That is expected: then choose Runtime → Run all again.")
        os.kill(os.getpid(), 9)
    print("✅ Installed.")

In [ ]:
# ===== Cell 2: connection to NEXUS (reads your private key from Colab's 🔑 Secrets) =====
NEXUS_SERVER = "https://nexus-t823.onrender.com"     # your NEXUS server on Render
NEXUS_VOICE_KEY = ""
try:
    from google.colab import userdata
    NEXUS_VOICE_KEY = (userdata.get("NEXUS_VOICE_KEY") or "").strip()
except Exception as e:
    print("ℹ️  No NEXUS_VOICE_KEY secret found (" + type(e).__name__ + ").")
if len(NEXUS_VOICE_KEY) >= 16:
    print("✅ Voice key loaded — NEXUS will connect automatically in Cell 5.")
else:
    NEXUS_VOICE_KEY = ""
    print("⚠️  No voice key yet, so NEXUS can't connect by itself (the test page still works).")
    print("   Add it: 🔑 key icon (left sidebar) → Add new secret → Name NEXUS_VOICE_KEY → paste the same key as VOICE_CLONE_KEY in Render → Notebook access ON → run this cell again.")

In [ ]:
# ===== Cell 3: your voice (saved in your Google Drive, so you upload it only once) =====
# Best clone: 10–20 s, only you talking, normal speed, quiet room, no music, phone ~20 cm from your mouth.
import os, shutil, subprocess
import soundfile as sf
from IPython.display import Audio, display

REPLACE_MY_VOICE = False      # set to True (then run this cell) to upload a new recording
REF = "/content/voice/my_voice.wav"
os.makedirs("/content/voice", exist_ok=True)
USE_DRIVE = False             # True = keep your voice in Google Drive (asks for Drive access once)
SAVED = None
try:
    if not USE_DRIVE: raise RuntimeError("Drive turned off")
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/NEXUS", exist_ok=True)
    SAVED = "/content/drive/MyDrive/NEXUS/my_voice.wav"
except Exception as e:
    print("ℹ️  Google Drive not connected (" + type(e).__name__ + "): you'll upload the recording each session.")

if SAVED and os.path.exists(SAVED) and not REPLACE_MY_VOICE:
    shutil.copy(SAVED, REF)
    print("✅ Using your voice saved in Google Drive (MyDrive/NEXUS/my_voice.wav).")
elif os.path.exists(REF) and not REPLACE_MY_VOICE:
    print("✅ Using the recording you already uploaded in this session.")
else:
    from google.colab import files
    print("Click 'Choose files' and pick your recording (WAV, MP3, M4A or a video file)…")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was chosen — run Cell 3 again.")
    name = next(iter(uploaded))
    src = os.path.join("/content/voice", "original_" + os.path.basename(name))
    shutil.move(name, src)
    # one clean mono WAV: remove rumble, trim silence at both ends, even loudness, at most 20 s
    clean = ("highpass=f=70,"
             "silenceremove=start_periods=1:start_threshold=-40dB:start_silence=0.1,areverse,"
             "silenceremove=start_periods=1:start_threshold=-40dB:start_silence=0.2,areverse,"
             "loudnorm=I=-20:TP=-2:LRA=11")
    r = subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", src, "-vn", "-ac", "1", "-ar", "24000", "-af", clean, "-t", "20", REF], capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(REF):
        raise SystemExit("❌ Could not read that file as audio: " + (r.stderr.strip()[-300:] or "unknown error") + "\\nExport it as WAV or MP3 and try again.")
    if SAVED:
        shutil.copy(REF, SAVED); print("💾 Saved in your Google Drive: MyDrive/NEXUS/my_voice.wav")
info = sf.info(REF)
print(f"✅ Voice ready: {info.duration:.1f} s, {info.samplerate} Hz")
if info.duration < 5:
    print("⚠️  Very short — 10–20 s of clear speech sounds much more like you. Set REPLACE_MY_VOICE = True and record again.")
display(Audio(REF))

In [ ]:
# ===== Cell 4: load the voice model and make a test sentence (first run downloads ~3 GB, 1–3 minutes) =====
import os, re, time
import numpy as np, torch, soundfile as sf
from IPython.display import Audio, display

USE_TURBO = False   # False = Chatterbox (closest to your voice, has the sliders). True = Chatterbox-Turbo (faster).
REF = "/content/voice/my_voice.wav"
if not os.path.exists(REF):
    raise SystemExit("❌ No voice recording yet — run Cell 3 first.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("⚠️  Running on the CPU (no GPU): this works but is slow.")

_loaded = globals().get("_nexus_loaded")      # running this cell again keeps the model already in GPU memory
try:
    if _loaded and _loaded[0] == USE_TURBO:
        model = _loaded[1]
    elif USE_TURBO:
        from chatterbox.tts_turbo import ChatterboxTurboTTS
        model = ChatterboxTurboTTS.from_pretrained(device=DEVICE)
    else:
        from chatterbox.tts import ChatterboxTTS
        model = ChatterboxTTS.from_pretrained(device=DEVICE)
    _nexus_loaded = (USE_TURBO, model)
except ModuleNotFoundError:
    raise SystemExit("❌ Chatterbox is not installed in this session. Runtime → Run all (Cell 1 installs it).")
except Exception as e:
    raise SystemExit(f"❌ The model could not be loaded: {e}\nCheck the internet connection of the notebook (it downloads from Hugging Face) and run Cell 4 again.")
SR = model.sr
print(f"✅ Model ready on {DEVICE.upper()} ({'Turbo' if USE_TURBO else 'Chatterbox'}), output {SR} Hz"
      + (f" — GPU: {torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else ""))

# ---- speed-ups that keep the exact same voice (same model, same settings) ----
# 1) Your recording is analysed ONCE here. (Passing it to every generate() call re-reads, resamples and re-encodes
#    the WAV for every single sentence.)
model.prepare_conditionals(REF, exaggeration=0.5) if not USE_TURBO else model.prepare_conditionals(REF)
# 2) Chatterbox asks the transformer for its attention maps on every token, but only the multilingual model reads
#    them. Asking for them forces the slow "eager" attention instead of the fast fused one (SDPA): same maths.
FAST = globals().get("FAST") or {"attn": True}
from chatterbox.models.t3.inference.t3_hf_backend import T3HuggingfaceBackend
if not getattr(T3HuggingfaceBackend, "_nexus_fast", False):
    _orig_forward = T3HuggingfaceBackend.forward
    def _fast_forward(self, *args, output_attentions=False, **kw):
        if FAST["attn"] and getattr(self, "alignment_stream_analyzer", None) is None:
            output_attentions = False
        return _orig_forward(self, *args, output_attentions=output_attentions, **kw)
    T3HuggingfaceBackend.forward = _fast_forward
    T3HuggingfaceBackend._nexus_fast = True

# 3) THE BIG ONE — the token loop. Each speech token runs the 30-layer transformer once, and PyTorch launches ~1,400
#    small GPU programs per token from Python; on Colab's CPU the launching was slower than the GPU work (GPU ~50 %
#    idle). A CUDA graph records the one-token step once and replays it with ONE launch. Same model, same sampling.
#    (torch.compile can't trace this transformers version on PyTorch 2.6, so the graph is recorded by hand.)
#    FAST["graph"] = False — or any error — uses Chatterbox's own loop instead.
FAST.setdefault("graph", True); FAST.setdefault("fp16", True)
import copy
from chatterbox.models.t3.t3 import T3
from transformers import StaticCache
from transformers.generation.logits_process import RepetitionPenaltyLogitsProcessor, TopPLogitsWarper, MinPLogitsWarper
if not hasattr(T3, "_nexus_orig"):
    T3._nexus_orig = T3.inference

class _GraphStep:
    """the transformer with a fixed-size cache; the one-token step is recorded once as a CUDA graph and replayed.
    FAST["fp16"]: the graph runs on a half-precision COPY of the transformer (T4 tensor cores, half the memory traffic).
    Measured on a T4 with your voice: fp32 graph 6.53 s → fp16 graph 4.35 s for the same 6.9 s of speech, voice
    similarity to your recording 0.927 → 0.936 (Chatterbox's own speaker encoder). FAST["fp16"] = False → fp32."""
    MAX_LEN = 1536                                        # prompt (~100–400) + up to ~1100 speech tokens (~45 s)
    def __init__(self, t3):
        self.half = bool(FAST.get("fp16", True))
        self.tf, self.head = (copy.deepcopy(t3.tfmr).half(), copy.deepcopy(t3.speech_head).half()) if self.half else (t3.tfmr, t3.speech_head)
        self.cache = StaticCache(config=self.tf.config, max_cache_len=self.MAX_LEN)
        self.graph = None
    def _fwd(self):
        return self.head(self.tf(inputs_embeds=self.emb, past_key_values=self.cache, cache_position=self.cp, use_cache=True).last_hidden_state)
    def prefill(self, embeds):
        embeds = embeds.half() if self.half else embeds
        self.cache.reset()                                # zeroes the same memory (the graph keeps pointing at it)
        L = embeds.shape[1]
        h = self.tf(inputs_embeds=embeds, past_key_values=self.cache, cache_position=torch.arange(L, device=embeds.device), use_cache=True).last_hidden_state
        if self.graph is None:                            # first sentence: record the step (writes only into slot L, which the real step overwrites)
            self.emb = torch.zeros((2, 1, embeds.shape[2]), dtype=embeds.dtype, device=embeds.device)
            self.cp = torch.full((1,), L, dtype=torch.long, device=embeds.device)
            s = torch.cuda.Stream(); s.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(s):
                for _ in range(2): self._fwd()
            torch.cuda.current_stream().wait_stream(s)
            g = torch.cuda.CUDAGraph()
            with torch.cuda.graph(g):
                self.out = self._fwd()
            self.graph = g
        return self.head(h[:, -1:]).float()
    def step(self, emb, position):
        self.emb.copy_(emb); self.cp.fill_(position)
        self.graph.replay()
        return self.out.float() if self.half else self.out

def _fast_t3_inference(self, **kw):
    cfg_weight = kw.get("cfg_weight", 0.5)
    if (not FAST.get("graph")) or self.device.type != "cuda" or getattr(self, "is_gpt", False) or self.hp.is_multilingual \
            or cfg_weight <= 0.0 or kw.get("initial_speech_tokens") is not None:
        return T3._nexus_orig(self, **kw)
    temperature, top_p, min_p = kw.get("temperature", 0.8), kw.get("top_p", 0.95), kw.get("min_p", 0.05)
    max_new = kw.get("max_new_tokens") or self.hp.max_speech_tokens
    text_tokens = torch.atleast_2d(kw["text_tokens"]).to(dtype=torch.long, device=self.device)
    start, stop = self.hp.start_speech_token, self.hp.stop_speech_token
    embeds, _ = self.prepare_input_embeds(t3_cond=kw["t3_cond"], text_tokens=text_tokens,
                                          speech_tokens=start * torch.ones_like(text_tokens[:, :1]), cfg_weight=cfg_weight)
    dev = embeds.device
    pos = self.speech_pos_emb.emb.weight                      # positions as a table: no per-token copy to the GPU
    ids = torch.full((1, max_new + 1), start, dtype=torch.long, device=dev)   # generated tokens (BOS first)
    bos = self.speech_emb(ids[:, :1]) + pos[0]
    prompt = torch.cat([embeds, torch.cat([bos, bos])], dim=1)
    try:
        g = getattr(self, "_nexus_graph", None) or _GraphStep(self); self._nexus_graph = g
        max_new = min(max_new, g.MAX_LEN - prompt.shape[1] - 1)
        logits_last = g.prefill(prompt); p = prompt.shape[1]
    except Exception as e:                                # never lose a sentence over a speed-up
        print("ℹ️  CUDA graph off (" + type(e).__name__ + ": " + str(e)[:120] + ") — using Chatterbox's own loop.")
        FAST["graph"] = False; self._nexus_graph = None
        return T3._nexus_orig(self, **kw)
    rep = RepetitionPenaltyLogitsProcessor(penalty=float(kw.get("repetition_penalty", 1.2)))
    minp = MinPLogitsWarper(min_p=min_p)
    topp = TopPLogitsWarper(top_p=top_p) if top_p < 1.0 else None
    cfg = torch.as_tensor(cfg_weight, device=dev, dtype=logits_last.dtype)
    n, CHECK = 1, 8
    for i in range(max_new):
        lg = logits_last[:, -1, :]
        cond, uncond = lg[0:1, :], lg[1:2, :]
        logits = cond + cfg * (cond - uncond)
        seen = ids[:, :n]
        logits = rep(seen, logits)
        if temperature != 1.0:
            logits = logits / temperature
        logits = minp(seen, logits)
        if topp is not None:
            logits = topp(seen, logits)
        nxt = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
        ids[:, n:n + 1] = nxt; n += 1
        if (i + 1) % CHECK == 0 and bool((ids[0, n - CHECK:n] == stop).any()):
            break
        emb = self.speech_emb(nxt) + pos[i + 1]
        emb = torch.cat([emb, emb])
        logits_last = g.step(emb, p + i)                 # read by the next sampling step, before the next replay
    toks = ids[:, 1:n]
    hit = (toks[0] == stop).nonzero()
    return toks[:, : int(hit[0]) + 1] if len(hit) else toks   # up to and including the first stop token

T3.inference = _fast_t3_inference
if getattr(model.t3, "_nexus_graph", None) is not None and not isinstance(model.t3._nexus_graph, _GraphStep):
    model.t3._nexus_graph = None                          # a graph from an older version of this cell

def split_sentences(text, max_chars=260):
    """Long text is spoken sentence by sentence (more stable), joined with short natural pauses."""
    text = re.sub(r"\s+", " ", text).strip()
    parts, cur = [], ""
    for s in re.split(r"(?<=[.!?])\s+", text):
        while len(s) > max_chars:                       # a very long sentence: break at a comma or space
            cut = max(s.rfind(",", 0, max_chars), s.rfind(" ", 0, max_chars))
            cut = cut if cut > 40 else max_chars
            parts.append(s[:cut + 1].strip()); s = s[cut + 1:].strip()
        if cur and len(cur) + 1 + len(s) > max_chars:
            parts.append(cur); cur = s
        else:
            cur = (cur + " " + s).strip()
    if cur: parts.append(cur)
    return [p for p in parts if p]

def speak(text, exaggeration=0.5, cfg_weight=0.5, temperature=0.8):
    """Text → (sample_rate, numpy audio) in your voice."""
    text = (text or "").strip()
    if not text:
        raise ValueError("Please type something to say.")
    if len(text) > 1200:
        raise ValueError("That is too long — please keep it under 1200 characters.")
    pieces, gap = [], np.zeros(int(SR * 0.18), dtype=np.float32)
    for part in split_sentences(text):
        with torch.inference_mode():
            if USE_TURBO:
                wav = model.generate(part, temperature=temperature)
            else:   # your voice was analysed once above; a changed "expressiveness" is applied without re-analysing
                wav = model.generate(part, exaggeration=exaggeration, cfg_weight=cfg_weight, temperature=temperature)
        pieces += [wav.squeeze(0).float().numpy(), gap]          # already on the CPU (no extra copy)
    audio = np.concatenate(pieces[:-1]) if pieces else np.zeros(1, dtype=np.float32)
    peak = float(np.max(np.abs(audio))) or 1.0
    return SR, (audio / peak * 0.89).astype(np.float32)      # clear and loud enough, never clipped

speak("Warming up.")          # first call: loads GPU kernels and records the CUDA graph — now, not on your first message
t0 = time.time()
try:
    sr, audio = speak("Hello! I am NEXUS, and this is my brand new voice. How do I sound?")
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); raise SystemExit("❌ The GPU ran out of memory. Runtime → Restart session, then run Cells 3 and 4 again.")
sf.write("/content/voice/test.wav", audio, sr)
print(f"✅ Test sentence made in {time.time() - t0:.1f} s — listen:")
display(Audio("/content/voice/test.wav"))

In [ ]:
# ===== Cell 5: the web page (public link) — type any English sentence and hear it in your voice =====
import secrets, tempfile, traceback, io, threading, time
import gradio as gr
import soundfile as sf

ACCESS_CODE = globals().get("ACCESS_CODE") or secrets.token_urlsafe(9)   # protects your free GPU: only people with the code can use it

def _check(code):
    if (code or "").strip() != ACCESS_CODE:
        raise gr.Error("Wrong access code — copy it from the output of Cell 5.")

GPU_LOCK = globals().get("GPU_LOCK") or threading.Lock()   # one sentence at a time on the GPU (web page and NEXUS share it)

def _to_file(sr, audio):
    f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False); f.close()   # system temp folder: Gradio may serve it
    sf.write(f.name, audio, sr); return f.name

def ui_generate(text, code, expressiveness, pace, randomness):
    _check(code)
    try:
        with GPU_LOCK:
            sr, audio = speak(text, exaggeration=expressiveness, cfg_weight=pace, temperature=randomness)
    except ValueError as e:
        raise gr.Error(str(e))
    except Exception as e:
        traceback.print_exc(); raise gr.Error(f"Could not make the audio: {e}")
    path = _to_file(sr, audio)
    return path, path

def api_speak(text, code):
    """Used by NEXUS (through your Render server): text + access code → WAV in your voice."""
    _check(code)
    try:
        with GPU_LOCK:
            sr, audio = speak(text)
    except ValueError as e:
        raise gr.Error(str(e))
    return _to_file(sr, audio)

with gr.Blocks() as demo:
    gr.Markdown("# 🎙️ NEXUS — my voice\nType an English sentence, press **Speak**, then play or download it.")
    code = gr.Textbox(label="Access code (printed under Cell 5)", type="password")
    text = gr.Textbox(label="What should I say?", lines=4, placeholder="Hi! I'm NEXUS. Nice to meet you.")
    with gr.Accordion("Voice settings (optional)", open=False):
        ex = gr.Slider(0.25, 1.0, value=0.5, step=0.05, label="Expressiveness (higher = more emotion)")
        cfg = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Pacing (lower = slower, calmer — try 0.3 if you speak fast)")
        temp = gr.Slider(0.5, 1.2, value=0.8, step=0.05, label="Variation (lower = steadier, higher = livelier)")
    btn = gr.Button("🔊 Speak in my voice", variant="primary")
    out = gr.Audio(label="Your voice", type="filepath")
    dl = gr.File(label="Download the WAV file")
    btn.click(ui_generate, [text, code, ex, cfg, temp], [out, dl], api_name="generate")
    # hidden: the API NEXUS calls ("speak")
    api_text = gr.Textbox(visible=False); api_code = gr.Textbox(visible=False); api_out = gr.Audio(visible=False, type="filepath")
    gr.Button(visible=False).click(api_speak, [api_text, api_code], api_out, api_name="speak")

demo.queue(default_concurrency_limit=1)          # one sentence at a time on the free GPU
_old = globals().get("_nexus_demo")
if _old is not None:                              # Cell 5 was run before: stop the old page first
    try: _old.close()
    except Exception: pass
_nexus_demo = demo
demo.launch(share=True, prevent_thread_lock=True, quiet=True, show_error=True)
PUBLIC_URL = demo.share_url

# ---- the fast lane NEXUS uses: ONE request → WAV bytes (the Gradio API needs 3 round trips through the public link
#      plus a temporary file). Same access code, same voice; the timing headers show where the time went. ----
from starlette.routing import Route
from starlette.responses import Response, JSONResponse
import anyio
async def _nexus_speak(request):
    t_in = time.perf_counter()
    try: body = await request.json()
    except Exception: return JSONResponse({"error": "Send JSON: {text, code}"}, status_code=400)
    if (str(body.get("code") or "")).strip() != ACCESS_CODE:
        return JSONResponse({"error": "Wrong access code"}, status_code=401)
    def run():
        with GPU_LOCK:
            t_gpu = time.perf_counter(); sr, audio = speak(str(body.get("text") or "")); t_done = time.perf_counter()
        buf = io.BytesIO(); sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")   # in memory, no temp file
        return buf.getvalue(), t_gpu - t_in, t_done - t_gpu, len(audio) / sr
    try:
        wav, waited, infer, secs = await anyio.to_thread.run_sync(run)
    except ValueError as e:
        return JSONResponse({"error": str(e)}, status_code=400)
    except Exception as e:
        traceback.print_exc(); return JSONResponse({"error": f"Could not make the audio: {e}"}, status_code=500)
    total = time.perf_counter() - t_in
    return Response(wav, media_type="audio/wav", headers={
        "x-nexus-queue-ms": str(round(waited * 1000)), "x-nexus-infer-ms": str(round(infer * 1000)),
        "x-nexus-total-ms": str(round(total * 1000)), "x-nexus-audio-ms": str(round(secs * 1000)), "cache-control": "no-store"})
try:
    demo.app.router.routes.insert(0, Route("/nexus/speak", _nexus_speak, methods=["POST"]))
except Exception as e:
    print("ℹ️  Fast lane not available (" + type(e).__name__ + "); NEXUS will use the normal Gradio API.")
print("\n" + "=" * 70)
if PUBLIC_URL:
    print("✅ Your voice page is live:", PUBLIC_URL)
else:
    print("⚠️  Colab could not create a public link right now (Gradio's share service). Run Cell 5 again in a minute.")
print("🔑 Access code:", ACCESS_CODE)
print("Keep this Colab tab open — your voice works only while this notebook is running.")
print("=" * 70)

# ---- connect to NEXUS automatically, and keep reminding it every minute (Render may restart while idle) ----
import threading, requests, time as _time
NEXUS_VOICE_KEY = globals().get("NEXUS_VOICE_KEY", ""); NEXUS_SERVER = globals().get("NEXUS_SERVER", "https://nexus-t823.onrender.com")
def _register_once():
    r = requests.post(f"{NEXUS_SERVER}/api/voice-clone/register", json={"url": PUBLIC_URL, "code": ACCESS_CODE},
                      headers={"x-nexus-voice-key": NEXUS_VOICE_KEY}, timeout=90)   # 90 s: a sleeping Render server takes time to wake
    return r.status_code, (r.json() if r.headers.get("content-type", "").startswith("application/json") else {})
def _heartbeat():
    while globals().get("_nexus_heartbeat_id") == _my_id:
        try: _register_once()
        except Exception: pass
        _time.sleep(60)
if NEXUS_VOICE_KEY and PUBLIC_URL:
    try:
        st, body = _register_once()
        if st == 200: print("🤖 Connected to NEXUS — it now speaks in your voice.")
        elif st == 401: print("❌ NEXUS refused the voice key: NEXUS_VOICE_KEY here must equal VOICE_CLONE_KEY in Render.")
        elif st == 409: print("❌ NEXUS isn't set to your voice yet: in Render add TTS_PROVIDER = clone.")
        elif st == 404: print("❌ Your NEXUS server is an older version without this feature — publish the latest NEXUS update.")
        else: print("⚠️  NEXUS answered", st, body.get("error", ""))
    except Exception as e:
        print("⚠️  Could not reach NEXUS yet (" + type(e).__name__ + ") — it will keep trying every minute.")
    _my_id = _time.time(); globals()["_nexus_heartbeat_id"] = _my_id
    threading.Thread(target=_heartbeat, daemon=True).start()
else:
    print("ℹ️  Not connected to NEXUS automatically (no voice key in Cell 2). The test page above still works.")

In [ ]:
# ===== Cell 6: test the voice exactly the way NEXUS uses it =====
import json, requests
from IPython.display import Audio, display
base = (PUBLIC_URL or "").rstrip("/")
if not base:
    raise SystemExit("Run Cell 5 first (it creates the public link).")
def call_speak(text):
    for prefix in ("/gradio_api", ""):
        r = requests.post(f"{base}{prefix}/call/speak", json={"data": [text, ACCESS_CODE]}, timeout=30)
        if r.status_code == 404: continue
        r.raise_for_status(); event_id = r.json()["event_id"]
        sse = requests.get(f"{base}{prefix}/call/speak/{event_id}", timeout=180).text
        if "event: complete" not in sse:
            raise RuntimeError("The voice API answered with an error — look at the output of Cell 5.")
        data = json.loads(sse.split("event: complete", 1)[1].split("data:", 1)[1].strip().splitlines()[0])
        return requests.get(data[0]["url"], timeout=60).content
    raise RuntimeError("No 'speak' API found — run Cell 5 again.")
wav = call_speak("This is my voice, coming to NEXUS through the API.")
print(f"✅ Your voice API works ({len(wav) // 1024} KB). Listen:")
display(Audio(wav))
try:
    st = requests.get(f"{globals().get('NEXUS_SERVER', 'https://nexus-t823.onrender.com')}/api/status", timeout=90).json().get("speech", {}).get("tts", {})
    print("NEXUS says:", "✅ " if st.get("configured") else "⏳ ", st.get("note", "?"))
except Exception as e:
    print("Could not ask NEXUS for its status right now:", type(e).__name__)

In [ ]:
# ===== Cell 7 (optional): speed check — Chatterbox's original path vs this notebook's, same text, same seeds =====
# Takes ~1 minute. Prints where the time goes and how close each result sounds to your recording.
import time, statistics, subprocess, numpy as np, torch, soundfile as sf, librosa
from IPython.display import Audio, display
BENCH_TEXT = "Hello! I am NEXUS, your robot companion. Today I can help you learn about space, science and history."
RUNS = 3
sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
ref_emb = model.ve.embeds_from_wavs([librosa.load(REF, sr=16000)[0]], sample_rate=16000).mean(0)
def similarity(a, sr):
    """voice similarity to your recording (cosine of Chatterbox's own speaker embeddings; 1.0 = identical voice)"""
    e = model.ve.embeds_from_wavs([librosa.resample(np.asarray(a, dtype=np.float32), orig_sr=sr, target_sr=16000)], sample_rate=16000).mean(0)
    return float(np.dot(e, ref_emb) / (np.linalg.norm(e) * np.linalg.norm(ref_emb)))
T = {}
def _timed(name, fn):
    def w(*a, **k):
        sync(); t = time.perf_counter(); out = fn(*a, **k); sync()
        T[name] = T.get(name, 0.0) + time.perf_counter() - t
        if name == "t3": T["tokens"] = T.get("tokens", 0) + int(out.shape[-1])
        return out
    return w
model.prepare_conditionals = _timed("voice", model.prepare_conditionals); model.t3.inference = _timed("t3", model.t3.inference)
model.s3gen.inference = _timed("s3gen", model.s3gen.inference); model.watermarker.apply_watermark = _timed("wm", model.watermarker.apply_watermark)
def original_path(text):
    """how Chatterbox runs out of the box: the recording re-analysed per sentence, its own token loop, fp32"""
    pieces = []
    for part in split_sentences(text):
        with torch.inference_mode():
            pieces.append(model.generate(part, audio_prompt_path=REF, exaggeration=0.5, cfg_weight=0.5, temperature=0.8).squeeze(0).float().numpy())
    return SR, np.concatenate(pieces)
def bench(label, fn, fast):
    saved = dict(FAST); FAST.update(fast); rows = []
    try:
        for i in range(RUNS):
            T.clear(); torch.manual_seed(1234 + i)
            p = subprocess.Popen(["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits", "-lms", "200"], stdout=subprocess.PIPE, text=True)
            t = time.perf_counter(); sr, a = fn(BENCH_TEXT); sync(); tot = time.perf_counter() - t; p.terminate()
            util = [int(x) for x in p.stdout.read().split() if x.isdigit()]
            rows.append(dict(total=tot, audio=len(a) / sr, sim=similarity(a, sr), gpu=statistics.mean(util) if util else -1, **T))
    finally:
        FAST.clear(); FAST.update(saved)
    m = {k: statistics.mean(r.get(k, 0) for r in rows) for k in rows[0]}
    print(f"\n{label}  (mean of {RUNS} runs)")
    print(f"  total {m['total']:.2f} s for {m['audio']:.1f} s of speech · real-time factor {m['total']/m['audio']:.2f} · GPU busy {m['gpu']:.0f} %")
    print(f"  voice analysis {m.get('voice',0):.2f} s · speech tokens {m['t3']:.2f} s ({m['tokens']/m['t3']:.0f} tokens/s) · vocoder {m['s3gen']:.2f} s · watermark {m['wm']:.2f} s")
    print(f"  voice similarity to your recording {m['sim']:.3f} · runs: " + ", ".join(f"{r['total']:.2f}s" for r in rows))
    return m, a
speak("Warm up.")
m0, a0 = bench("BEFORE — Chatterbox's original path", original_path, {"attn": False, "graph": False})
m1, a1 = bench("AFTER — this notebook (voice analysed once, CUDA graph, fp16 transformer)", speak, {})
print(f"\n⏱  {m0['total']:.2f} s → {m1['total']:.2f} s for the same text ({m0['total']/m1['total']:.1f}× faster) · similarity {m0['sim']:.3f} → {m1['sim']:.3f}")
sf.write("/content/voice/before.wav", a0, SR); sf.write("/content/voice/after.wav", a1, SR)
print("Listen — before, then after:"); display(Audio("/content/voice/before.wav")); display(Audio("/content/voice/after.wav"))
for obj, name in ((model, "prepare_conditionals"), (model.t3, "inference"), (model.s3gen, "inference"), (model.watermarker, "apply_watermark")):
    obj.__dict__.pop(name, None)                      # remove the timers again


## Free-tier limits (good to know)
- **The GPU is free but limited:** Colab gives free users a T4 GPU for a few hours at a time (no fixed promise). If it says no GPU is available, try later.
- **Sessions stop:** after about 90 minutes without using the notebook, or at most about 12 hours, Colab shuts the session down. Then your voice page and NEXUS's voice stop until you open the notebook and **Run all** again.
- **The public link changes every session** — you don't need to do anything: the notebook tells NEXUS the new link itself (and again every minute). While the notebook is off, NEXUS reports the voice as unavailable; turn on *Settings → Voice → "If the voice service fails, speak with this computer's voice"* so NEXUS is never silent.
- **Speed (measured on a free T4, Cell 7):** 6.9 s of speech takes ~4.3 s to make (Chatterbox out of the box: ~8.8 s). NEXUS asks for each sentence as soon as the AI has written it, so the first sentence starts while the rest are still being made.
- **Your recording stays in this Colab session** (it is deleted when the session ends). It is never put in your GitHub repo or on the NEXUS website.

## If something goes wrong
- *"No module named chatterbox"* → Runtime → Run all (Cell 1 installs and restarts once).
- *"CUDA out of memory"* → Runtime → Restart session, then Cells 3, 4, 5.
- *The voice does not sound like you* → record 15–20 seconds in a quiet room, phone 20 cm from your mouth, speaking naturally; upload again in Cell 3, then run Cell 4. In the page, try Pacing 0.3–0.4 and Expressiveness 0.4–0.6.
- *The page says "Wrong access code"* → copy the code printed under Cell 5 again.